# 03 · Dois algoritmos disputam o modelo preditivo

**Etapa 4.** Os dois candidatos usam **exatamente as mesmas features eleitas** no notebook 02, o **mesmo treino** e são avaliados na **mesma validação temporal** (últimos 20% do ciclo), pela métrica principal **MAE**.

| | Algoritmo | Racional |
|---|---|---|
| **Modelo A** | `Ridge` (regressão linear L2) | baseline interpretável, rápido, robusto a colinearidade |
| **Modelo B** | `RandomForestRegressor` | captura não linearidades (perfil horário × dia útil × clima) |

Cada candidato é um `Pipeline` (imputador + `StandardScaler` + modelo) com `fit` **apenas no treino**; o pipeline serializado leva consigo o pré-processamento e a lista de features.

In [1]:
# Setup: garante caminhos consistentes (nbconvert executa com cwd = notebooks/)
import os, sys, warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
import mlflow

from src import utils
cfg = utils.carregar_config()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: /home/runner/work/bsd/bsd


In [2]:
import joblib
from src import preprocessamento as pp, modelagem

ciclo = utils.carregar_ciclo()
DIR_EV = utils.dir_evidencias(ciclo["cycle_id"])
sel = utils.ler_json(utils.DIR_PROCESSED / "features_eleitas.json")
assert sel["cycle_id"] == ciclo["cycle_id"], "Votação não pertence ao ciclo atual"
eleitas = sel["features_eleitas"]

dados = pd.read_parquet(utils.DIR_PROCESSED / "dataset_ciclo.parquet")
treino, val = pp.separar_treino_validacao(dados, cfg)
X_treino, y_treino = pp.xy(treino, cfg, eleitas)
X_val, y_val = pp.xy(val, cfg, eleitas)
print(ciclo["cycle_id"], "| features:", eleitas)
print("treino:", X_treino.shape, "| validação:", X_val.shape)

ciclo_3 | features: ['is_rush_hour', 'hr_sin', 'atemp', 'temp', 'hr_cos', 'yr', 'hum', 'weathersit', 'hr', 'temp_hum_interaction', 'season', 'workingday', 'mnth_cos']
treino: (12735, 13) | validação: (3184, 13)


In [3]:
exp_id = utils.configurar_mlflow(cfg)
DIR_CAND = utils.DIR_PROCESSED / "candidatos"
DIR_CAND.mkdir(parents=True, exist_ok=True)
resultados, previsoes_val = [], {}

for papel, spec in cfg["modelos"].items():
    algoritmo, params = spec["algoritmo"], spec["params"]
    pipe, metricas = modelagem.treinar_e_avaliar(eleitas, algoritmo, params, X_treino, y_treino, X_val, y_val)
    previsoes_val[algoritmo] = modelagem.prever(pipe, X_val)
    caminho = DIR_CAND / f"{ciclo['cycle_id']}__{algoritmo}.joblib"
    joblib.dump(pipe, caminho, compress=3)

    with mlflow.start_run(run_name=f"{ciclo['cycle_id']}_03_{papel}_{algoritmo}",
                          tags={**utils.tags_ciclo(ciclo, "03_treino"), "papel": papel, "algoritmo": algoritmo}) as run:
        mlflow.log_params({f"modelo__{k}": v for k, v in params.items()})
        mlflow.log_params({
            "cycle_id": ciclo["cycle_id"], "data_version": ciclo["data_version"], "algoritmo": algoritmo,
            "papel": papel, "seed": cfg["projeto"]["seed"], "features": ",".join(eleitas),
            "n_features": len(eleitas), "n_treino": len(X_treino), "n_validacao": len(X_val),
            "protocolo": "holdout temporal 80/20", "metrica_principal": cfg["metrica"]["principal"],
            "run_id_votacao": sel["run_id_votacao"],
        })
        mlflow.log_metrics(metricas)
        info = mlflow.sklearn.log_model(pipe, name="modelo", input_example=X_val.head(5),
                                        skops_trusted_types=["numpy.dtype", "sklearn.tree._tree.Tree"])
    resultados.append({"papel": papel, "algoritmo": algoritmo, "params": params, "features": eleitas,
                       "metricas": metricas, "run_id": run.info.run_id, "model_uri": info.model_uri,
                       "arquivo_candidato": utils.relativo(caminho)})
    print(f"{papel:9s} {algoritmo:22s} MAE val = {metricas['val_mae']:.3f} | run {run.info.run_id}")

modelo_a  Ridge                  MAE val = 109.711 | run 9f6e6f15898a4745b04d56315b0d1998


modelo_b  RandomForestRegressor  MAE val = 51.624 | run 97f2a37685fd468c982bc4cfc0208a76


## Comparação dos candidatos (mesmo protocolo)

In [4]:
comparacao = pd.DataFrame([{"papel": r["papel"], "algoritmo": r["algoritmo"], **r["metricas"], "run_id": r["run_id"]}
                           for r in resultados]).set_index("algoritmo")
comparacao.to_csv(DIR_EV / "competicao_modelos.csv")
display(comparacao.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 3.8), gridspec_kw={"width_ratios": [3, 1]})
janela = val.tail(24 * 10)
axes[0].plot(janela["dteday"] + pd.to_timedelta(janela["hr"], unit="h"), janela["cnt"], color="black", lw=1, label="real")
for alg, p in previsoes_val.items():
    axes[0].plot(janela["dteday"] + pd.to_timedelta(janela["hr"], unit="h"), p[-len(janela):], lw=1, label=alg)
axes[0].set_title("Validação – últimos 10 dias"); axes[0].legend(fontsize=8)
comparacao["val_mae"].plot.bar(ax=axes[1], color=["tab:orange", "tab:green"]); axes[1].set_title("MAE validação")
axes[1].tick_params(axis="x", rotation=0, labelsize=8)
fig.tight_layout(); fig.savefig(DIR_EV / "competicao_validacao.png", dpi=110); plt.show()

utils.salvar_json({"cycle_id": ciclo["cycle_id"], "data_version": ciclo["data_version"],
                   "features": eleitas, "resultados": resultados},
                  utils.DIR_PROCESSED / "resultado_competicao.json")

,papel,val_mae,val_rmse,val_r2,treino_mae,treino_rmse,treino_r2,tempo_treino_s,run_id
algoritmo,,,,,,,,,
Ridge,modelo_a,109.711,137.775,0.643,58.256,80.814,0.741,0.032,9f6e6f15898a4745b04d56315b0d1998
RandomForestRegressor,modelo_b,51.624,79.465,0.881,20.338,32.687,0.958,0.914,97f2a37685fd468c982bc4cfc0208a76


PosixPath('/home/runner/work/bsd/bsd/data/processed/resultado_competicao.json')